In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import sklearn
import math
import statistics
import warnings
warnings.filterwarnings(action='ignore')
pd.options.display.float_format = '{:.5f}'.format
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

temp_data=pd.read_csv("/Users/kbh/Downloads/lending_club_2020_train.csv",low_memory=False)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)

In [3]:
temp_data.drop(956843,axis=0,inplace=True)

In [4]:
temp_data.shape

(1755294, 141)

In [5]:
temp_data['term']=temp_data['term'].str[:3].astype(int)
temp_data['int_rate']=temp_data['int_rate'].str[:6].astype(float)
temp_data=temp_data.replace({'emp_length':'< 1 year'},'0.5',)
temp_data['emp_length']=temp_data.emp_length.str.extract(r'(\d\d|\d+.\d+|\d)').astype(float)
temp_data['last_pymnt_d'] = pd.to_datetime(temp_data['last_pymnt_d'], format='%b-%Y').dt.to_period('M')
temp_data['last_credit_pull_d'] = pd.to_datetime(temp_data['last_credit_pull_d'], format='%b-%Y').dt.to_period('M')

라벨 인코딩

In [7]:
labelencoding=LabelEncoder()
temp_data.sub_grade=labelencoding.fit_transform(temp_data['sub_grade'].astype(str))

NA 행을 제거 하기로 한거(기존)

In [9]:
eliminate_list='''annual_inc
delinq_2yrs
earliest_cr_line
inq_last_6mths
open_acc
pub_rec
total_acc
acc_now_delinq
tot_coll_amt
tot_cur_bal
delinq_amnt
mo_sin_old_rev_tl_op
mo_sin_rcnt_rev_tl_op
mo_sin_rcnt_tl
num_accts_ever_120_pd
num_actv_bc_tl
num_actv_rev_tl
num_bc_tl
num_il_tl
num_op_rev_tl
num_rev_accts
num_rev_tl_bal_gt_0
num_tl_30dpd
num_tl_90g_dpd_24m
num_tl_op_past_12m
tot_hi_cred_lim
total_il_high_credit_limit
last_credit_pull_d
'''
## last_credit_pull_d << 추가됨
eliminate_list = eliminate_list.strip().split('\n')


NA를 0으로 대체

In [11]:
replace_zero_list='''mths_since_last_delinq
annual_inc_joint
dti_joint
verification_status_joint
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_cu_tl
inq_last_12m
acc_open_past_24mths
revol_bal_joint
sec_app_fico_range_low
sec_app_fico_range_high
sec_app_earliest_cr_line
sec_app_inq_last_6mths
sec_app_mort_acc
sec_app_open_acc
sec_app_revol_util
sec_app_open_act_il
sec_app_num_rev_accts
sec_app_chargeoff_within_12_mths
sec_app_collections_12_mths_ex_med
inq_fi
emp_length
'''
replace_zero_list = replace_zero_list.strip().split('\n')

열 삭제

In [13]:
eliminate_col='''emp_title
url
title
zip_code
addr_state
mths_since_last_record
next_pymnt_d
hardship_flag
hardship_type
hardship_reason
hardship_status
deferral_term
hardship_amount
hardship_start_date
hardship_end_date
payment_plan_start_date
hardship_length
hardship_dpd
hardship_loan_status
hardship_payoff_balance_amount
hardship_last_payment_amount
orig_projected_additional_accrued_interest
grade'''
eliminate_col = eliminate_col.strip().split('\n')

temp_data.drop(eliminate_col,axis=1,inplace=True)

날짜 데이터 변환

In [15]:
# temp_data['payment_plan_start_date']=pd.to_datetime(temp_data['payment_plan_start_date'], format='%b-%Y').dt.to_period('M')
temp_data['issue_d']=pd.to_datetime(temp_data['issue_d'], format='%b-%Y').dt.to_period('D')
temp_data['earliest_cr_line']=pd.to_datetime(temp_data['earliest_cr_line'], format='%b-%Y').dt.to_period('D')

# 위에 없는 NA 값 중에서 본인이 맡았던 것 해주시면 됩니다.

병현

In [18]:
idx_na_mths_since_recent_bc=temp_data[temp_data['mths_since_recent_bc'].isnull()].index
mths_since_recent_bc_median=temp_data['mths_since_recent_bc'].median()
temp_data.loc[idx_na_mths_since_recent_bc,'mths_since_recent_bc']=mths_since_recent_bc_median
idx_na_mths_since_recent_bc=temp_data[temp_data['mths_since_recent_inq'].isnull()].index
mths_since_recent_bc_median=temp_data['mths_since_recent_inq'].median()
temp_data.loc[idx_na_mths_since_recent_bc,'mths_since_recent_inq']=mths_since_recent_bc_median
temp_data['revol_util']=temp_data['revol_util'].str.replace('%', '').astype(float)
eliminate_list.append('dti')

하영

In [20]:
##### 이거 NAN 처리 지금은 안해요~
# last_pymn_d : 돈을 한번도 안낸 사람들이 모두 결측치다 => NAN 처리 안함
# 1. No Payment 나 '2000-01' 형태의 날짜로 결측치 채우기
temp_data['last_pymnt_d'].fillna(pd.to_datetime('2000-01', format='%Y-%m').to_period('M'), inplace=True)
# 2. issue_d 로 값을 채워넣는다 (돈을 안낸 사람은 모두. 발행일이다 )( X 불가 같은 날인 사람들 있음)
## 상환을 받았어야 하는 d-day 로 부터 지난 날 얼마나 지났는지 ? =>


##### 확정
# charge_off = 0 : 모두 2007 년에 issue : 12 개월 훨씬 전이기에, 0 으로 채워넣는다
temp_data['chargeoff_within_12_mths'].fillna(0, inplace=True)

#1. bc_util
#1.1 : 분모 값인 total_rev_hi_lim 이 nan 값(이거 drop하기로함) 제거
#1.2 : 나머지를 가지고 분포 확인 -> 원래 데이터와 분포가 비슷
#1.3 : 나머지는 mean 일때 분포가 비슷했기때문에, mean 으로 대체
mean_bc_util = temp_data['bc_util'].mean()
temp_data['bc_util'].fillna(mean_bc_util, inplace=True)

#### 확정 x

###2. il_util
#### => 이거 아직 ~
## 2.1 total_bal_il (남은 내야 할 . 돈( 분자값 ) 결측치 -> 0 으로 처리, 이것도 0 처리 )
## 2.2 분포 일치하는 것을 찾지 못함 -> 분모/ 분자 값으로 계산
## 2.3 이 과정에서, 분모가 0 인값은 총 7개로, 삭제를 권장..
#total_bal_il 이거 결측치가 519721 개 인데, 0으로 처리하기로함 ==> 이거 먼저 실행해야함
# 그러고도 남은 nan 값은 75%값 or ....
numerator = temp_data['total_bal_il']
denominator = temp_data['total_il_high_credit_limit']
temp_data['il_util'] = temp_data['il_util'].fillna(numerator / denominator)


지환

In [22]:
# last_credit_pull_d : na 행을 제거, 위에 코드에 변수 추가해놓은 상태

# mo_sin_old_il_acct : 중앙값으로 대체
idx_na_mo_sin_old_il_acct = temp_data[temp_data['mo_sin_old_il_acct'].isnull()].index
mo_sin_old_il_acct_median = temp_data['mo_sin_old_il_acct'].median()
temp_data.loc[idx_na_mo_sin_old_il_acct, 'mo_sin_old_il_acct'] = mo_sin_old_il_acct_median

# pct_tl_nvr_dlq : 부트스트래핑
idx_na_pct_tl_nvr_dlq = temp_data[temp_data['pct_tl_nvr_dlq'].isnull()].index  # 결측치가 있는 인덱스를 저장
pct_tl_nvr_dlq_non_na = temp_data['pct_tl_nvr_dlq'].dropna()  # 결측치가 없는 값을 저장

np.random.seed(42)  # 시드 설정
pct_tl_nvr_dlq_bootstrap_samples = np.random.choice(pct_tl_nvr_dlq_non_na, size=len(idx_na_pct_tl_nvr_dlq), replace=True)  # 부트스트래핑으로 샘플 생성

temp_data.loc[idx_na_pct_tl_nvr_dlq, 'pct_tl_nvr_dlq'] = pct_tl_nvr_dlq_bootstrap_samples  # 생성된 값으로 결측치 대체


# pub_rec_bankruptcies: linear regression (pub_rec가 독립변수)

현범

In [24]:
mths1_med=temp_data['mths_since_recent_bc_dlq'].median()
mths2_med=temp_data['mths_since_recent_revol_delinq'].median()
temp_data['mths_since_recent_bc_dlq'].fillna(mths1_med, inplace=True)
temp_data['mths_since_recent_revol_delinq'].fillna(mths2_med, inplace=True)
replace_zero_list.append('revol_util')
replace_zero_list.append('bc_open_to_buy')
eliminate_col.append('num_sats')

한직

In [26]:
# 총 5개, 결측치

# mths_since_last_delinq : 결측치 수(922,199)
## 차용인의 마지막 연체 이후 경과한 개월 수, 연체한 적이 없다고 판단되어, “연체 이후 경과한 개월 수”를 0으로 설정.
## 결측치에 해당하는 loan_status 확인시, loan_status 전체 분포 대비 특이사항 없음.
replace_zero_list.append('mths_since_last_delinq')
# mths_since_last_major_derog : 결측치 수(1,321,878)
## 마지막 주요 불이행 후 경과된 개월 수, 결측치가 있는 경우를 “중대한 불이행이 없었다”로 해석하여, 이를 0으로 설정
## 결측치에 해당하는 loan_status 확인시, loan_status 전체 분포 대비 특이사항 없음.
replace_zero_list.append('mths_since_last_major_derog')
# num_bc_sats # 현재 보유하고 있는 신용카드의 수 : (35,113)
## num_bc_sats 값이 결측치인 경우에도 대출이 상환된 비율(Fully Paid)이 상당히 높음. 따라서, 결측치가 반드시 부정적인 결과와 연관되어 있지 않음을 시사할수 있음.
## 평균값이 약 4.85, 중앙값은 4, 최대값은 77 인점(이상치) 감안할때, 신용카드의 갯수를 평균값보다 중앙값으로 설정
median_value = temp_data['num_bc_sats'].median()
temp_data['num_bc_sats'].fillna(median_value, inplace=True)

# num_tl_120dpd_2m # 최근 2개월 동안 120일 이상 연체된 계좌의 수 : (96,751)
## Fully Paid: 63,734개로 전체 결측치 데이터 대비 높음(결측치가 있는 데이터 중 상당수가 Fully Paid 상태에 있으므로, 결측치가 반드시 부정적인 결과와 연관되어 있다고 단정할 수는 없음)
## 피처의 대부분의 값이 0임을 고려할 때, 결측치를 0으로 채우는 것이 합리적임.
replace_zero_list.append('num_tl_120dpd_2m')

# tax_liens # 세금 유치권의 수 : (73개)
## 결측치의 분포상, loan_status가 Does not meet the credit policy인 경우가 많고, 그 중 상당수가 Fully Paid 상태임.
## tax_liens 피처의 결측치는 세금 관련 압류권이 없었기 때문에 발생했을 가능성이 높음.
## 대부분의 값이 0이고 tax_liens 값의 25%, 50%, 75% 분포가 모두 0이므로, 결측치를 0으로 채우는 것이 데이터의 분포와 일치할 가능성이 높음.
replace_zero_list.append('tax_liens')

준희

* mths_since_last_record : 변수 제거 (이미 위에서 제거 완료)
* collections_12_mths_ex_med : 행 제거
* percent_bc_gt_75: 부트스트랩 샘플링, 신용등급을 기준으로 나눠서
* total_bal_ex_mort: 부트스트랩 샘플링, 신용등급을 기준으로 나눠서
* total_bc_limit : 부트스트랩 샘플링, 신용등급을 기준으로 나눠서
* mort_acc : 부트스트랩 샘플링, 신용등급을 기준으로 나눠서

In [29]:

# 1. null 값인 행 제거
eliminate_list.append('collections_12_mths_ex_med')

# 2. 부트스트랩 샘플링을 통한 대체 함수 정의
def bootstrap_imputation(df, column, groupby_column):
    # 그룹별로 결측치를 대체
    for group in df[groupby_column].unique():
        group_data = df[df[groupby_column] == group]
        non_null_values = group_data[column].dropna().values

        if len(non_null_values) > 0:
            # 결측치에 대해 부트스트랩 샘플링을 적용
            sampled_values = np.random.choice(non_null_values, size=group_data[column].isna().sum(), replace=True)
            df.loc[(df[groupby_column] == group) & (df[column].isna()), column] = sampled_values

    return df

# 3. percent_bc_gt_75 결측치 대체
temp_data = bootstrap_imputation(temp_data, 'percent_bc_gt_75', 'sub_grade')

# 4. total_bal_ex_mort 결측치 대체
temp_data = bootstrap_imputation(temp_data, 'total_bal_ex_mort', 'sub_grade')

# 5. total_bc_limit 결측치 대체
temp_data = bootstrap_imputation(temp_data, 'total_bc_limit', 'sub_grade')

# 6. mort_acc 결측치 대체
temp_data = bootstrap_imputation(temp_data, 'mort_acc', 'sub_grade')



In [30]:
temp_data.loc[(temp_data['num_il_tl'].isna() |
               temp_data['total_bal_il'].isna() |
               temp_data['all_util'].isna()) &
    temp_data['il_util'].isna(),'il_util']=0

In [31]:
temp_data.loc[(temp_data['num_il_tl'].isna() |
               temp_data['total_bal_il'].isna() |
               temp_data['all_util'].isna()) &
    temp_data['il_util'].isna(),'il_util'].head()

Series([], Name: il_util, dtype: float64)

In [32]:
temp_data = bootstrap_imputation(temp_data, 'il_util', 'sub_grade')

# 최종 실행

In [34]:
eliminate_index=[]
for i in eliminate_list:
    eliminate_index.append(temp_data[temp_data[i].isnull()].index.astype(int))
eliminate_index= list(set(item for sublist in eliminate_index for item in sublist))
temp_data.drop(eliminate_index,axis=0,inplace=True)

In [35]:
for i in replace_zero_list:
    temp_data[i].fillna(0, inplace=True)

In [36]:
temp_data.dropna(inplace=True)

In [37]:
temp_data.isnull().sum()

id                                    0
loan_amnt                             0
funded_amnt                           0
funded_amnt_inv                       0
term                                  0
int_rate                              0
installment                           0
sub_grade                             0
emp_length                            0
home_ownership                        0
annual_inc                            0
verification_status                   0
issue_d                               0
loan_status                           0
pymnt_plan                            0
purpose                               0
dti                                   0
delinq_2yrs                           0
earliest_cr_line                      0
fico_range_low                        0
fico_range_high                       0
inq_last_6mths                        0
mths_since_last_delinq                0
open_acc                              0
pub_rec                               0


In [38]:
fico_zero_data = temp_data[temp_data['last_fico_range_high'] == 0]

for sub_grade in fico_zero_data['sub_grade'].unique():
    sub_grade_data = temp_data[(temp_data['sub_grade'] == sub_grade) & (temp_data['last_fico_range_high'] > 0)]

    if not sub_grade_data.empty:
        sampled_values = np.random.choice(sub_grade_data['last_fico_range_high'], size=len(fico_zero_data[fico_zero_data['sub_grade'] == sub_grade]), replace=True)
        temp_data.loc[(temp_data['sub_grade'] == sub_grade) & (temp_data['last_fico_range_high'] == 0), 'last_fico_range_high'] = sampled_values

temp_data['last_fico_range_high'].value_counts()
temp_data.drop(columns=['last_fico_range_low'], inplace=True)


idx_num_bc_tl = temp_data.loc[temp_data['num_bc_tl'] > 61].index
temp_data = temp_data.drop(idx_num_bc_tl)
idx_annual_inc = temp_data.loc[temp_data['annual_inc'] > 9999999.00].index
temp_data_filtered = temp_data.drop(idx_annual_inc)

In [39]:
temp_data.shape

(1711211, 117)

In [40]:
cu_in_data=temp_data.loc[temp_data['loan_status']=='Current',:]
cu_out_data=temp_data.loc[temp_data['loan_status']!='Current',:]

In [41]:
cu_in_data.to_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_in_data.csv',index=False)
cu_out_data.to_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_out_data.csv',index=False)

In [42]:
temp_current=pd.read_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_in_data.csv')
temp_out=pd.read_csv('/Users/kbh/Downloads/lending club/all_clean_current_final_out_data.csv')

In [77]:
temp_out.head()

,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,last_fico_range_high,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,debt_settlement_flag
0,124989905,6000.00000,6000.00000,6000.00000,36,7.97000,187.94000,4,2.00000,MORTGAGE,45000.00000,Verified,2017-12-01,Fully Paid,n,debt_consolidation,8.67000,1.00000,2005-09-01,755.00000,759.00000,0.00000,22.00000,14.00000,0.00000,3090.00000,14.00000,36.00000,w,0.00000,0.00000,6718.84000,6718.84000,6000.00000,718.84000,0.00000,0.00000,0.00000,2020-04,1835.06000,2020-04,764.00000,0.00000,0.00000,1.00000,Joint App,100500.00000,10.20000,Verified,0.00000,0.00000,50054.00000,1.00000,10.00000,3.00000,3.00000,3.00000,46964.00000,83.00000,1.00000,1.00000,3090.00000,64.00000,22100.00000,1.00000,0.00000,2.00000,4.00000,3850.00000,11910.00000,20.60000,0.00000,0.00000,147.00000,135.00000,12.00000,3.00000,1.00000,56.00000,37.00000,3.00000,33.00000,0.00000,1.00000,1.00000,1.00000,4.00000,28.00000,4.00000,7.00000,1.00000,14.00000,0.00000,0.00000,0.00000,4.00000,97.10000,0.00000,0.00000,0.00000,78611.00000,50054.00000,15000.00000,56511.00000,3926.00000,555.00000,559.00000,Sep-2005,0.00000,0.00000,10.00000,0.00000,10.00000,4.00000,1.00000,0.00000,N
1,84253847,23200.00000,23200.00000,23200.00000,60,24.99000,680.82000,23,10.00000,MORTGAGE,110000.00000,Verified,2016-07-01,Charged Off,n,debt_consolidation,34.70000,1.00000,1993-05-01,670.00000,674.00000,1.00000,10.00000,24.00000,0.00000,16909.00000,55.90000,60.00000,f,0.00000,0.00000,8599.68000,8599.68000,1030.43000,2309.25000,0.00000,5260.00000,946.80000,2016-12,680.82000,2019-02,639.00000,0.00000,0.00000,1.00000,Individual,0.00000,0.00000,0,0.00000,0.00000,606327.00000,1.00000,10.00000,1.00000,2.00000,10.00000,277552.00000,90.00000,4.00000,6.00000,4698.00000,85.00000,30255.00000,4.00000,3.00000,7.00000,8.00000,25264.00000,6014.00000,70.40000,0.00000,0.00000,126.00000,277.00000,6.00000,6.00000,2.00000,7.00000,10.00000,0.00000,10.00000,0.00000,8.00000,10.00000,8.00000,14.00000,36.00000,12.00000,22.00000,10.00000,24.00000,0.00000,0.00000,0.00000,5.00000,94.80000,37.50000,0.00000,0.00000,650914.00000,294461.00000,20300.00000,291465.00000,0.00000,0.00000,0.00000,0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,Y
2,119708428,16000.00000,16000.00000,16000.00000,36,7.07000,494.55000,1,0.00000,MO